## Gold — dim_municipio (DTB)
**Origem:** `workspace.silver.municipios` → **Destino:** `workspace.gold.dim_municipio`

- **Grão:** 1 linha por município (dimensão conformada, SCD1)
- **Transformações:** adiciona `sk_municipio` (surrogate key sequencial), garante ordenação por `codigo_municipio`, mantém apenas colunas conformadas, sem duplicatas de NK, pronta para join com fatos (ex.: CNO por `codigo_municipio`).
- **Modelo:** Star Schema — `dim_municipio` para uso em análises (qualidade por atributo + respostas às perguntas do Objetivo).
- **Linhagem:** `silver.municipios` (limpo) → `gold.dim_municipio` (dimensão).

In [0]:
%run ./_setup_dtb

In [0]:
from pyspark.sql import functions as F, Window
from data_pipeline import save_table, add_column_comments
from metadata.metadata import DIM_MUNICIPIO_COMMENTS

In [0]:
SOURCE_TABLE = "workspace.silver.municipios"
TARGET_TABLE = "workspace.gold.dim_municipio"

In [0]:
df = spark.table(SOURCE_TABLE)
print(f"Silver in: {df.count():,} | colunas={df.columns}")

# Garante NK válida e ordena para SK determinístico
df = df.filter(F.col("codigo_municipio").rlike("^[0-9]{7}$"))
w = Window.orderBy("codigo_municipio")
df = df.withColumn("sk_municipio", F.row_number().over(w).cast("int"))

# Reordena: SK primeiro, depois NK e atributos (silver agora só tem 5 cols: sem região)
ordered = ["sk_municipio", "codigo_municipio", "nome_municipio", "codigo_uf", "sigla_uf", "nome_uf"]
cols = [c for c in ordered if c in df.columns] + [c for c in df.columns if c not in ordered]
df = df.select(*cols)

display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE, mode="overwrite")
existing = set(spark.table(TARGET_TABLE).columns)
filtered = {k: v for k, v in DIM_MUNICIPIO_COMMENTS.items() if k in existing}
if filtered:
    add_column_comments(spark, TARGET_TABLE, filtered)

In [0]:
total = spark.table(TARGET_TABLE).count()
distinct_sk = spark.table(TARGET_TABLE).select("sk_municipio").distinct().count()
distinct_nk = spark.table(TARGET_TABLE).select("codigo_municipio").distinct().count()
print(f"Total: {total:,} | SK distintos: {distinct_sk:,} | NK distintos: {distinct_nk:,}")
assert total == distinct_sk == distinct_nk, "Quebra de unicidade SK/NK!"
display(spark.sql(f"SELECT sigla_uf, count(*) as qtd FROM {TARGET_TABLE} GROUP BY sigla_uf ORDER BY sigla_uf"))
display(spark.sql(f"SELECT nome_uf, count(*) as qtd FROM {TARGET_TABLE} GROUP BY nome_uf ORDER BY nome_uf"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))